# Knee osteoarthritis: binary classification with classical ML

This notebook is a **viewer** for the `kneeclf` package (`src/kneeclf/`). Training runs from the command line
(locally with `configs/quick.json`, on a server with `configs/full.json`) and writes everything to a run directory:
`metrics.json`, `comparison.csv`/`.md`, `params.json`, `model.pkl`, `train.log` and `figures/`.

- **Task**: Kellgren-Lawrence (KL) grades 0-4 are mapped to a binary label. Default `kl01_vs_2to4`: KL 0-1 = no OA,
  KL 2-4 = OA (the usual radiographic definition). The previous approach (`kl0_vs_1to4`) is still available,
  see [ADR-0002](../docs/adr/0002-binary-label-scheme.md).
- **Features and models**: raw pixels + PCA, and a numpy HOG descriptor, fed to logistic regression, RBF SVM,
  random forest and gradient boosting, each tuned by `GridSearchCV` (stratified folds, ROC-AUC).
- **Recall first**: the decision threshold is chosen on validation to reach a target sensitivity (default 0.85);
  the test set is used once, at the end, with everything frozen. See [ADR-0003](../docs/adr/0003-classical-features-and-recall-first-evaluation.md).

## Parameters

In [ ]:
from pathlib import Path

DATA_ROOT = Path("Dataset_Knee_Osteoarthritis")   # <root>/{train,val,test}/{0,1,2,3,4}/*.png (not in the repo)
RUNS = Path("../runs")                            # parent of run directories
RUN_DIR = None                                    # None = latest run in RUNS; or Path("../runs/2026...")
LAUNCH_QUICK = False                              # True = train with configs/quick.json first (about 15 s)

## Optional: launch the quick configuration

Small sample (150 images per class), 64 px, tiny grids. Only a smoke test: its numbers are not results.

In [ ]:
import subprocess, sys, os

if LAUNCH_QUICK:
    env = {**os.environ, "PYTHONPATH": str(Path("../src").resolve())}
    cmd = [sys.executable, "-m", "kneeclf", "--data", str(DATA_ROOT.resolve()),
           "--config", str(Path("../configs/quick.json").resolve()), "--output", str(RUNS.resolve())]
    print(" ".join(cmd))
    subprocess.run(cmd, env=env, check=True)

## Load a run

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

if RUN_DIR is None:
    runs = sorted(p for p in RUNS.glob("*") if (p / "metrics.json").exists())
    assert runs, f"no finished run under {RUNS.resolve()}"
    RUN_DIR = runs[-1]
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
print("run:", RUN_DIR)
print("label scheme:", metrics["config"]["label_scheme"], "| image size:", metrics["config"]["image_size"])
print("environment:", metrics["environment"])

### Data profile

Train is balanced by undersampling (if `balance_train`); validation and test keep their real class ratio.

In [ ]:
pd.DataFrame({s: {"files": p["files_found"], "kept": p["kept"], **{f"label {k}": v for k, v in p["labels_kept"].items()},
                  "MB": p["memory_mb"]} for s, p in metrics["data_profile"].items()}).T

### Candidate comparison (validation)

Each candidate gets its own threshold tuned on validation; the ranking uses validation ROC-AUC, which does not depend on the threshold.

In [ ]:
comparison = pd.read_csv(RUN_DIR / "comparison.csv")
comparison

### Selected model

In [ ]:
sel = metrics["selected"]
print(f"{sel['candidate']}  params={sel['best_params']}  threshold={sel['threshold']:.4f} ({sel['threshold_mode']}, target recall {sel['target_recall']})")
pd.DataFrame({"validation": metrics["validation"], "test": metrics["test"]}).drop(["confusion_matrix"])

### Test set (touched once)

Recall = sensitivity (OA cases found); specificity = healthy knees correctly cleared.

In [ ]:
for f in ["test_curves.png", "test_confusion.png", "val_curves.png", "val_confusion.png"]:
    p = RUN_DIR / "figures" / f
    if p.exists():
        display(Markdown(f"**{f}**"), Image(filename=str(p)))

### Timings

In [ ]:
pd.Series(metrics["timings"], name="seconds").to_frame()

## Reading the results

- A high recall with a low specificity means the threshold buys sensitivity with false alarms: compare the
  confusion matrices of validation and test to check the threshold transferred.
- KL 1 ("doubtful") is radiographically close to both KL 0 and KL 2: most errors are expected at that border.
- These models see global texture and edges, not the joint space width a radiologist measures; they are
  baselines, not a diagnostic tool.